# 05 · Normalization & stationarity

Features become comparable and model-ready through **transforms** (`sabia.normalize.*`). A transform factory returns a `BoundTransform` whose `.apply(expr) -> expr` you pipe onto a feature expression — same causality invariant, its own pinned spec.

The headline concept is **fractional differentiation** (López de Prado 2018): prices are non-stationary, but ordinary differencing (returns) throws away almost all *memory*. Fractional differencing recovers stationarity while keeping most of the signal — a genuine trade-off you can dial with one parameter. We also show the trailing `zscore` and cross-sectional `xs_rank`. Requires `pip install plotly nbformat jupyterlab`.

In [1]:
import sys, pathlib
HERE = pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent))

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"

import sabia

import _simulate as sim
import _stats as st
from _data import make_ohlcv

SCHEMA = sim.default_schema()

# Transforms are separate from features but obey the same causality invariant: a BoundTransform's
# .apply(expr) -> expr pipes onto a feature expression (or a plain column).
frame = make_ohlcv(n=750)         # a non-stationary random-walk close
close = frame.get_column("close").to_numpy()
print(f"ADF t-stat of raw close: {st.adf_tstat(close):.2f}  (5% critical value ~ -2.86 -> non-stationary)")

ADF t-stat of raw close: -2.65  (5% critical value ~ -2.86 -> non-stationary)


## Fractional differentiation: the trade-off, measured
We sweep `d` from 0 to 1, and for each measure stationarity (a Dickey-Fuller t-stat) and memory (correlation with the original price). The whole story is in this little table.

In [2]:
# Fractional differentiation (Lopez de Prado 2018): difference by a *fractional* order d in [0,1].
# d=0 is the raw price (max memory, non-stationary); d=1 is the ordinary return (stationary, memory
# destroyed). Fractional d in between buys stationarity while preserving most of the memory.
ds = [round(0.1 * k, 1) for k in range(0, 11)]
cols = {f"d={d}": sabia.normalize.frac_diff(d).apply(pl.col("close")).alias(f"ffd_{d}")
        for d in ds if d > 0}
ffd = frame.select("timestamp", "close", *cols.values())

records = []
for d in ds:
    if d == 0.0:
        series = close
    else:
        series = ffd.get_column(f"ffd_{d}").to_numpy()
    mask = ~np.isnan(series)
    adf = st.adf_tstat(series[mask])
    corr = float(np.corrcoef(series[mask], close[mask])[0, 1])
    records.append((d, adf, corr))
grid = pl.DataFrame(records, schema=["d", "adf", "corr_with_price"], orient="row")
grid

d,adf,corr_with_price
f64,f64,f64
0.0,-2.648718,1.0
0.1,-3.552366,0.982539
0.2,-5.159962,0.923369
0.3,-7.227164,0.819962
0.4,-9.671738,0.684014
…,…,…
0.6,-15.138677,0.401708
0.7,-17.916159,0.288757
0.8,-20.614364,0.201846


## …seen on the series
The raw price wanders (a unit root); its `d=0.3` fractional difference is stationary yet still rises and falls *with* the price — memory the return (`d=1`) would have erased.

In [3]:
t = ffd.get_column("timestamp").to_list()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=t, y=ffd.get_column("close").to_list(), name="close (d=0, non-stationary)",
                         line=dict(color="#90a4ae", width=1.4)), secondary_y=False)
fig.add_trace(go.Scatter(x=t, y=ffd.get_column("ffd_0.3").to_list(), name="frac_diff d=0.3 (stationary)",
                         line=dict(color="#d62728", width=1.2)), secondary_y=True)
fig.update_yaxes(title_text="close", secondary_y=False)
fig.update_yaxes(title_text="fractionally differenced", secondary_y=True)
fig.update_layout(height=420,
                  title="A wandering price vs its d=0.3 fractional difference — stationary, but still tracks the swings",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
                  margin=dict(l=60, r=30, t=90, b=40))
fig.show()

## …and as a curve
As `d` grows the series gets more stationary (ADF drops) but less correlated with the price. The practical choice is the **smallest** `d` that clears the stationarity threshold — here a low `d` already does, retaining most of the memory.

In [4]:
# The trade-off curve: stationarity (ADF, more negative = better) vs memory (correlation with the
# original price). The sweet spot is the *smallest* d whose ADF clears the threshold.
d_arr = grid.get_column("d").to_list()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=d_arr, y=grid.get_column("adf").to_list(), name="ADF t-stat",
                         line=dict(color="#1f77b4", width=2), mode="lines+markers"), secondary_y=False)
fig.add_trace(go.Scatter(x=d_arr, y=grid.get_column("corr_with_price").to_list(), name="corr with price",
                         line=dict(color="#2ca02c", width=2, dash="dot"), mode="lines+markers"),
              secondary_y=True)
fig.add_hline(y=-2.86, line=dict(color="#d62728", width=1.2, dash="dash"),
              annotation_text="5% stationarity threshold", secondary_y=False)
fig.update_xaxes(title_text="differencing order d")
fig.update_yaxes(title_text="ADF t-stat (more negative = stationary)", secondary_y=False)
fig.update_yaxes(title_text="correlation with original price", secondary_y=True, range=[0, 1.02])
fig.update_layout(height=440,
                  title="Stationarity vs memory — a small d already clears the threshold while keeping most memory",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
                  margin=dict(l=60, r=30, t=90, b=40))
fig.show()

## Trailing z-score
A rolling standardizer puts an indicator on a comparable scale over time. Piped straight onto the `rsi` feature expression.

In [5]:
# Rolling z-score: a trailing standardizer (no grouping needed on a single series). Apply it to a
# feature expression to put an unbounded indicator on a comparable scale.
rsi = sabia.momentum.rsi(period=14)
z = sabia.normalize.zscore(window=63)
ts = frame.select(
    "timestamp",
    rsi.expr(SCHEMA),
    z.apply(rsi.expr(SCHEMA)).alias("rsi_z63"),
).tail(300)
tt = ts.get_column("timestamp").to_list()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=("RSI(14) — bounded [0,100]", "…rolling z-score, zscore(63)"))
fig.add_trace(go.Scatter(x=tt, y=ts.get_column("rsi_14").to_list(), line=dict(color="#6a3d9a", width=1.2),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=tt, y=ts.get_column("rsi_z63").to_list(), line=dict(color="#1f77b4", width=1.2),
                         showlegend=False), row=2, col=1)
for y in (-2, 0, 2):
    fig.add_hline(y=y, row=2, col=1, line=dict(color="#455a64", width=0.7, dash="dot"))
fig.update_layout(height=480, title="Standardizing a feature with a trailing z-score",
                  margin=dict(l=60, r=30, t=80, b=40))
fig.show()

## Cross-sectional rank
Within each timestamp, rank a signal across the universe into `[0,1]` — the building block of cross-sectional models.

In [6]:
# Cross-sectional rank: standardize a signal *within each timestamp* across the universe. The robust
# pattern is to materialize the feature first, then rank that column (one .over(...) at a time).
SYMBOLS = ("AAA", "BBB", "CCC", "DDD", "EEE", "FFF")
panel = sim.momentum_panel(n=400, symbols=SYMBOLS)
mom = sabia.momentum.mom(formation=126, skip=21)
with_signal = panel.lazy().with_columns(mom.expr(SCHEMA)).collect()
xr = sabia.normalize.xs_rank(over="timestamp")
ranked = with_signal.select(
    "timestamp", "symbol", mom.spec.name,
    xr.apply(pl.col(mom.spec.name)).alias("mom_xs_rank"),
)
last_ts = panel.get_column("timestamp").max()
snap = ranked.filter(pl.col("timestamp") == last_ts).sort("mom_xs_rank")
fig = go.Figure(go.Bar(x=snap.get_column("symbol").to_list(), y=snap.get_column("mom_xs_rank").to_list(),
                       marker_color=snap.get_column("mom_xs_rank").to_list(), marker_colorscale="RdYlGn",
                       text=[f"{v:.2f}" for v in snap.get_column("mom_xs_rank").to_list()],
                       textposition="outside"))
fig.update_layout(height=380,
                  title="Cross-sectional rank of momentum on the final date — 0 = laggard, 1 = leader",
                  yaxis_title="xs_rank", yaxis_range=[0, 1.1], margin=dict(l=60, r=30, t=80, b=40))
fig.show()

## Takeaways

- **Fractional differentiation** turns a non-stationary price into a stationary, memory-preserving feature — pick the smallest `d` that passes a stationarity test.
- `zscore` (trailing) and `xs_rank` / `xs_zscore` (cross-sectional) standardize features for models without peeking ahead.
- Transforms carry their own fingerprinted spec, so a `FeatureSetManifest` pins them exactly alongside the features (see `examples/07_manifest.py`).